# Task 1

## 1. What is leave-one-out? Provide limitations and strengths

Крайний случай K-Fold, где k = n (число объектов). На каждой итерации в тест уходит ровно один объект, а модель обучается на остальных n-1. Итого n обучений, метрика усредняется по всем объектам.
### Сильные стороны

Максимально возможный размер обучающей выборки на каждом шаге → минимальное смещение (bias) оценки качества. Оценка близка к качеству модели, обученной на всём датасете.

Полностью детерминирован — нет случайного разбиения, нет random_state, результат воспроизводим всегда.

Использует каждый объект и как тест, и как трейн → нет «потери» данных.

Практически незаменим на очень маленьких выборках (n < 50–100), где отрезать 20 % на тест — роскошь.

### Ограничения

Вычислительная стоимость O(n) обучений. Для наших ~49 000 объявлений это 49 000 обучений — неприемлемо (для LinearRegression есть аналитический трюк, но для деревьев/бустинга — нет).

Высокая дисперсия оценки: тест из одного объекта даёт метрику либо «очень хорошо», либо «очень плохо»; для MSE/MAE усреднение спасает, а вот для accuracy на одном объекте — это 0 или 1.

Обучающие подвыборки почти идентичны друг другу (различаются одним объектом) → ошибки моделей сильно скоррелированы, поэтому LOO плохо оценивает разброс и склонен недооценивать ошибку для нестабильных моделей.

# 2. Grid Search, Randomized Search, Bayesian Optimization

## Grid Search (полный перебор по сетке)

Задаём дискретный список значений для каждого гиперпараметра, берём декартово произведение и обучаем модель во всех точках (обычно с CV).

Плюсы: прост, воспроизводим, легко параллелится, гарантированно найдёт лучшую точку внутри сетки.
Минусы: экспоненциальный рост — 3 параметра по 10 значений = 1000 запусков × k фолдов. Тратит бюджет на неважные параметры; разрешение по важному параметру ограничено шагом сетки.

## Randomized Search (случайный поиск)

Задаём распределения (uniform, log-uniform, choice) и берём n_iter случайных точек.

Ключевая идея (Bergstra & Bengio, 2012): обычно качество зависит от 1–2 параметров из 5. Grid тратит повторы на неважные оси, random — нет. При n_iter точках каждая координата пробуется в n_iter разных значениях, а не в 10.
Вероятность попасть в топ-5 % области параметров хотя бы раз = 1 - 0.95^n_iter; при n_iter=60 это ≈ 95 %.
Плюсы: бюджет задаётся явно, легко параллелится, хорошо работает в высокой размерности, не привязан к шагу сетки.
Минусы: не использует информацию о прошлых испытаниях — «слепой».

## Bayesian Optimization (TPE / GP — то, что внутри Optuna)

Последовательная (адаптивная) оптимизация:

1. Строим surrogate-модель (вероятностную аппроксимацию) функции «гиперпараметры → метрика»: Gaussian Process, Random Forest (SMAC) или TPE (Tree-structured Parzen Estimator, дефолт в Optuna).
2. Считаем acquisition function (например Expected Improvement), которая балансирует exploitation (искать там, где уже хорошо) и exploration (искать там, где мало данных / высокая неопределённость).
3. Берём аргмакс acquisition → это следующая точка для запуска. Обучаем модель, получаем реальную метрику, обновляем surrogate. Повторяем.

Плюсы: находит хороший оптимум за в разы меньшее число обучений; умеет условные/иерархические пространства; в Optuna есть pruning (досрочная остановка бесперспективных trial'ов).
Минусы: последовательность → хуже параллелится; есть свои гипер-гиперпараметры; на очень шумной CV-метрике может «переобучиться» на шум; накладные расходы на surrogate (мелочь по сравнению с обучением модели).

## Итого: Grid — когда параметров 1–2 и они дискретные; Random — надёжный baseline при фиксированном бюджете; Bayesian — когда одно обучение дорогое и параметров много.

# 3. Классификация методов отбора признаков

### (a) Filter-методы — оценивают признаки независимо от модели, по статистике «признак ↔ таргет» (или свойствам самого признака).


Примеры: доля NaN, нулевая/низкая дисперсия, корреляция Пирсона / Спирмена, χ², mutual information, ANOVA F-test, отбрасывание сильно скоррелированных пар.
Плюсы: очень быстро, масштабируется на десятки тысяч признаков, не переобучается.
Минусы: игнорирует взаимодействия и мультиколлинеарность (кроме специальных приёмов), «слепы» к тому, какая модель будет использоваться.

### (b) Wrapper-методы — перебирают подмножества признаков, обучая модель и меряя качество на валидации.


Примеры: Forward/Backward selection, RFE (Recursive Feature Elimination), exhaustive search, permutation importance (по духу — wrapper/оценка post-hoc).
Плюсы: учитывают взаимодействия и специфику модели, обычно дают лучший итоговый набор.
Минусы: дорого (много обучений), риск переобучиться на валидацию.

### (c) Embedded-методы — отбор встроен в процесс обучения.


Примеры: Lasso / ElasticNet (L1-регуляризация зануляет веса), деревья и их feature_importances_, градиентный бустинг, Elastic Net path.
Плюсы: почти бесплатно (одно обучение), учитывает мультиколлинеарность.
Минусы: привязано к семейству модели; для L1 нужна нормировка признаков; при коррелированных признаках выбор внутри группы нестабилен.

### (d) Hybrid / post-hoc интерпретация — SHAP, permutation importance, drop-column importance. Формально это методы объяснения, но их значения используют как критерий отбора.

### Корреляция Пирсона 

$$𝑟=\frac{∑(𝑥_𝑖−\overset{-}{x})(𝑦_𝑖−\overset{-}{y})}{\sqrt{∑(𝑥_𝑖−\overset{-}{x})^2}\sqrt{∑(𝑦_𝑖−\overset{-}{𝑦})^2}}=\frac{cov(𝑥,𝑦)}{𝜎_𝑥𝜎_𝑦}$$

Измеряет линейную связь, 
$𝑟∈[−1,1]$. Отбор: берём признаки с максимальным ∣𝑟∣ к таргету; дополнительно выкидываем один из пары признаков с $∣𝑟_{𝑥_𝑖𝑥_𝑗}∣>0.9$ (борьба с мультиколлинеарностью). Значимость проверяется t-статистикой $𝑡=𝑟\sqrt{(𝑛−2)/(1−𝑟^2)}$.
Ограничения: не видит нелинейных зависимостей (например, 𝑦=𝑥^2 даёт 𝑟≈0), чувствителен к выбросам, требует численных признаков.


### χ² (chi-squared)

Тест независимости для категориальных признака и таргета (в sklearn chi2 требует неотрицательные признаки — счётчики / бинарные / one-hot). Строим таблицу сопряжённости, считаем

$$𝜒^2=∑_{𝑖,𝑗}\frac{(𝑂_{𝑖𝑗}−𝐸{𝑖𝑗})^2}{𝐸_{𝑖𝑗}}$$



$𝐸_{𝑖𝑗}=\frac{𝑅_𝑖𝐶_𝑗}{𝑁}$ 

где 𝑂 — наблюдаемые частоты, 𝐸 — ожидаемые при независимости. Чем больше 𝜒2 (меньше p-value), тем сильнее признак связан с таргетом. Для регрессии таргет надо предварительно бинаризовать/разбить на квантили (у нас — цена в бины).

Ограничения: только классификация, только неотрицательные признаки, ненадёжен при малых ожидаемых частотах (<5), не измеряет силу эффекта, а только значимость (на больших n «значимо» почти всё).

### Lasso (L1-регуляризация). 
Минимизируется ‖y − Xw‖² + α‖w‖₁. Негладкий L1-штраф с «угловыми» линиями уровня приводит к тому, что часть коэффициентов становится ровно нулём → автоматический отбор признаков и разреженная модель. α управляет силой: чем больше α, тем меньше признаков остаётся (подбирается по CV, LassoCV). Важно масштабировать признаки. Ограничения: при сильно скоррелированных признаках выбирает произвольный один из группы (решается ElasticNet), при p ≫ n отбирает не более n признаков, оценки коэффициентов смещены.


### Permutation importance / significance. 
Обучаем модель, фиксируем базовую метрику на валидации. Затем случайно перемешиваем значения одного признака (разрушая его связь с y, сохраняя распределение) и заново измеряем метрику. Важность = падение качества, усреднённое по нескольким перестановкам. Значимость оценивается как статистическая: распределение метрик при перестановках даёт нулевое распределение, и p-value = доля перестановок, где качество не хуже исходного. Плюсы: model-agnostic, считается на hold-out (в отличие от impurity importance деревьев, смещённой к признакам с большим числом уникальных значений), не требует переобучения. Минусы: скоррелированные признаки «делят» важность и оба выглядят неважными; перемешивание создаёт нереалистичные объекты; результат стохастичен.


### SHAP (SHapley Additive exPlanations). 
Подход к интерпретации из кооперативной теории игр: предсказание раскладывается аддитивно на вклады признаков, f(x) = φ₀ + Σ φᵢ, где φ₀ — базовое значение (среднее предсказание), а φᵢ — значение Шепли признака i, то есть его средний маржинальный вклад по всем возможным порядкам добавления признаков в «команду». Это единственная аддитивная атрибуция, удовлетворяющая свойствам локальной точности, отсутствия влияния (missingness) и согласованности.


Реализации: TreeSHAP (быстрый, точный для деревьев/бустингов), KernelSHAP (model-agnostic, приближённый), DeepSHAP/GradientSHAP (нейросети), LinearSHAP.
Локальная интерпретация: waterfall / force plot — почему модель выдала такой прогноз для конкретного объекта.
Глобальная: mean(|SHAP|) как важность признаков, beeswarm (summary) plot — направление и величина влияния, dependence plot — форма зависимости и взаимодействия.
Плюсы: единая локальная + глобальная интерпретация, теоретические гарантии, знак и величина влияния.
Минусы: вычислительно дорого (кроме TreeSHAP), зависимость от выбора background-выборки, проблемы при скоррелированных признаках (маргинализация даёт нереалистичные комбинации), SHAP показывает влияние на предсказание модели, а не причинность.

In [591]:
import numpy as np
import pandas as pd
from collections import Counter
from itertools import product
from sklearn.model_selection import (KFold, GroupKFold, StratifiedKFold,
                                     TimeSeriesSplit, ParameterGrid, ParameterSampler)
from sklearn.linear_model import (Lasso, ElasticNet, LinearRegression)
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
import shap
import time
import warnings
import optuna
warnings.filterwarnings('ignore')

In [592]:
RANDOM_STATE = 21

In [593]:
train = pd.read_json("data/train.json")

In [594]:
def clean_value(value):
    for ch in ["[", "]", "'", '"', ' ']:
        value = value.replace(ch, '')
    return value

def clean_features_column(df):
    df = df.copy()
    df['features'] = df['features'].apply(
        lambda lst: [clean_value(f) for f in lst if clean_value(f)]
    )
    return df

def normalize(f):
    return f.lower()

def get_top_features(df, n=30):
    counter = Counter()
    canonical = {}
    for lst in df['features']:
        for f in lst:
            key = normalize(f)
            counter[key] += 1
            if key not in canonical:
                canonical[key] = f
    top_keys = [k for k, _ in counter.most_common(n)]
    return [canonical[k] for k in top_keys]

def make_binary_features(df, top_features):
    df = df.copy()
    top_norm = {normalize(f) for f in top_features}
    canon = {normalize(f): f for f in top_features}

    def row_to_norm_set(lst):
        return {normalize(f) for f in lst}

    sets = df['features'].apply(row_to_norm_set)
    for f in top_features:
        key = normalize(f)
        df[f] = sets.apply(lambda s, k=key: int(k in s))
    return df

train = clean_features_column(train)
TOP20 = get_top_features(train, 20)
train = make_binary_features(train, TOP20)
print(TOP20)

['Elevator', 'HardwoodFloors', 'CatsAllowed', 'DogsAllowed', 'Doorman', 'Dishwasher', 'LaundryinBuilding', 'NoFee', 'FitnessCenter', 'LaundryinUnit', 'Pre-War', 'RoofDeck', 'OutdoorSpace', 'DiningRoom', 'HighSpeedInternet', 'balcony', 'SwimmingPool', 'NewConstruction', 'Terrace', 'Exclusive']


In [595]:
lower = train['price'].quantile(0.01)
upper = train['price'].quantile(0.99)

train = train[(train['price'] >= lower) & (train['price'] <= upper)]
train = train[train['bathrooms'] <= 20]

In [596]:
train['created'] = pd.to_datetime(train['created'])

data = train[['created', 'price', 'manager_id', 'building_id',
              'bedrooms', 'bathrooms'] + TOP20].reset_index(drop=True)

TARGET, DATE = 'price', 'created'
X = data[TOP20]
y = data[TARGET]


In [597]:
def _take(obj, idx):
    """Position-based indexing that works for DataFrame/Series/ndarray"""
    return obj.iloc[idx] if hasattr(obj, "iloc") else obj[idx]

def _shuffled_indices(n, random_state=RANDOM_STATE, shuffle=True):
    idx = np.arange(n)
    if shuffle:
        np.random.RandomState(random_state).shuffle(idx)
    return idx

In [598]:
def train_test_split_custom(X, y, test_size=0.25, random_state=RANDOM_STATE, shuffle=True):
    """
    Randomly split into train / test.

    test_size : float in (0, 1) — fraction of rows going to the test part.
    Returns X_train, X_test, y_train, y_test.
    """

    if not 0.0 < test_size < 1.0:
        raise ValueError("test_size must be in (0, 1)")

    n = len(X)
    if n != len(y):
        raise ValueError("X and y must have the same number of rows")

    idx = _shuffled_indices(n, random_state, shuffle)

    n_test = int(round(n * test_size))
    n_test = max(1, min(n - 1, n_test))

    test_idx, train_idx = idx[:n_test], idx[n_test:]

    return (_take(X, train_idx), _take(X, test_idx),
            _take(y, train_idx), _take(y, test_idx))

In [599]:
def train_val_test_split_custom(X, y, validation_size=0.2, test_size=0.2, random_state=RANDOM_STATE, shuffle=True):
    """
    Randomly split into train / validation / test.

    validation_size, test_size : fractions OF THE WHOLE dataset.
    Returns X_train, X_val, X_test, y_train, y_val, y_test.
    """
    if not 0.0 < validation_size < 1.0:
        raise ValueError("validation_size must be in (0, 1)")
    if not 0.0 < test_size < 1.0:
        raise ValueError("test_size must be in (0, 1)")
    if validation_size + test_size >= 1.0:
        raise ValueError("validation_size + test_size must be < 1")

    n = len(X)
    if n != len(y):
        raise ValueError("X and y must have the same number of rows")

    idx = _shuffled_indices(n, random_state, shuffle)

    n_test = int(round(n * test_size))
    n_val = int(round(n * validation_size))
    n_test = max(1, n_test)
    n_val = max(1, n_val)
    if n_test + n_val >= n:
        raise ValueError("Not enough rows for the requested proportions")

    test_idx = idx[:n_test]
    val_idx = idx[n_test:n_test + n_val]
    train_idx = idx[n_test + n_val:]

    return (_take(X, train_idx), _take(X, val_idx), _take(X, test_idx),
            _take(y, train_idx), _take(y, val_idx), _take(y, test_idx))

In [600]:
def _prepare(df, date_column, target_column, feature_columns):
    """Preprocessing: feature list, parsed & sorted dates, NAT check"""
    if feature_columns is None:
        feature_columns = [c for c in df.columns if c not in (date_column, target_column)]

    dates = pd.to_datetime(df[date_column])
    if dates.isna().any():
        raise ValueError(f"Column '{date_column}' contains unparseable/NAT values")

    order = dates.sort_values(kind="mergesort").index
    df = df.loc[order]
    dates = dates.loc[order]

    return df, dates, feature_columns

In [601]:
def train_test_split_by_date(df, date_split, date_column=DATE,
                             target_column=TARGET, feature_columns=None):
    """
    Time-based split into train / test.

    date <  date_split -> train
    date >= date_split -> test
    Guarantees: max(train_date) < date_split <= max(test_date), both parts non-empty.
    """
    df, dates, feature_columns = _prepare(df, date_column, target_column, feature_columns)

    date_split = pd.to_datetime(date_split)
    
    if not (dates.min() < date_split <= dates.max()):
        raise ValueError(
            f"date_split={date_split} is outside the data range "
            f"[{dates.min()}, {dates.max()}]")

    mask_train = dates < date_split
    df_train = df.loc[mask_train]
    df_test = df.loc[~mask_train]

    return (df_train[feature_columns], df_test[feature_columns],
            df_train[target_column], df_test[target_column])

In [602]:
def train_val_test_split_by_date(df, validation_date, test_date,
                                 date_column=DATE, target_column=TARGET,
                                 feature_columns=None):
    """
    Time-based split into train / validation / test.

                    date <  validation_date -> train
    validation_date <= date <  test_date    -> validation
          test_date <= date                 -> test

    Guarantees (all parts non-empty):
        max(train) < validation_date <= max(val) < test_date <= max(test)
    """
    df, dates, feature_columns = _prepare(df, date_column, target_column, feature_columns)

    validation_date = pd.to_datetime(validation_date)
    test_date = pd.to_datetime(test_date)

    if validation_date >= test_date:
        raise ValueError("Validation date must be earlier than test_date")

    m_train = dates < validation_date
    m_val = (dates >= validation_date) & (dates < test_date)
    m_test = dates >= test_date

    for name, mask in (("train", m_train), ("validation", m_val), ("test", m_test)):
        if not mask.any():
            raise ValueError(
                f"'{name}' split is empty for validation_date={validation_date.date()}, "
                f"test_date={test_date.date()} and data range "
                f"[{dates.min()}, {dates.max()}]")

    parts = [df.loc[m] for m in (m_train, m_val, m_test)]
    return (*[p[feature_columns] for p in parts],
            *[p[target_column] for p in parts])

### Determinism of a split — what does it mean?

A split procedure is deterministic if calling it with the same inputs and the same parameters always produces exactly the same partition — the same rows in the same order, in the same run, in another run, on another machine, and on another day.

In [603]:
def kfold_split(X, k=5, shuffle=True, random_state=RANDOM_STATE):
    """
    Classic K-Fold.

    Parameters
    ----------
    X : array-like / DataFrame — only len(X) is used
    k : int, number of folds (2 <= k <= n)
    shuffle : bool
    random_state : int

    Returns
    -------
    list of (train_idx, test_idx) — np.ndarray of positional indices, len == k
    """
    n = len(X)
    if not isinstance(k, (int, np.integer)) or k < 2:
        raise ValueError("k must be an integer >= 2")
    if k > n:
        raise ValueError(f"k={k} cannot exceed n_samples={n}")

    indices = np.arange(n)
    if shuffle:
        np.random.RandomState(random_state).shuffle(indices)

    fold_sizes = np.full(k, n // k, dtype=int)
    fold_sizes[: n % k] += 1

    folds, start = [], 0
    for size in fold_sizes:
        test_idx = indices[start:start + size]
        train_idx = np.concatenate([indices[:start], indices[start + size:]])
        folds.append((train_idx, test_idx))
        start += size
    return folds


In [604]:
def group_kfold_split(df, k=5, group_field='manager_id'):
    """
    Grouped K-Fold: all rows sharing the same group value stay in one fold.

    Guarantees:
      - every group is entirely in train or entirely in test for each fold;
      - test parts are disjoint and cover all rows;
      - every fold is non-empty;
      - fully deterministic (no random_state needed).

    Returns list of (train_idx, test_idx) with POSITIONAL indices -> use .iloc
    """
    n = len(df)
    if not isinstance(k, (int, np.integer)) or k < 2:
        raise ValueError("k must be an integer >= 2")
    if group_field not in df.columns:
        raise ValueError(f"column '{group_field}' not found")

    inverse, uniq = pd.factorize(df[group_field], use_na_sentinel=False)
    inverse = np.asarray(inverse)
    n_groups = len(uniq)
    if k > n_groups:
        raise ValueError(f"k={k} > number of groups={n_groups}")

    counts = np.bincount(inverse, minlength=n_groups)

    order = np.argsort(-counts, kind='stable')
    fold_load = np.zeros(k, dtype=int)
    group_to_fold = np.empty(n_groups, dtype=int)
    for g in order:
        f = int(np.argmin(fold_load))
        group_to_fold[g] = f
        fold_load[f] += counts[g]

    row_fold = group_to_fold[inverse]

    indices = np.arange(n)
    folds = [(indices[row_fold != f], indices[row_fold == f]) for f in range(k)]

    assert sum(len(te) for _, te in folds) == n
    assert all(len(te) > 0 for _, te in folds)
    return folds

In [605]:
gfolds = group_kfold_split(data, k=5, group_field='manager_id')

print(f"{'fold':>4} {'train':>7} {'test':>6} {'test %':>7} {'managers':>9}")
for i, (tr, te) in enumerate(gfolds):
    print(f"{i:>4} {len(tr):>7} {len(te):>6} {len(te)/len(data):>6.1%} "
          f"{data['manager_id'].iloc[te].nunique():>9}")

leaks = [len(set(data['manager_id'].iloc[tr]) & set(data['manager_id'].iloc[te]))
         for tr, te in gfolds]
print("\nmanagers shared between train and test:", leaks)

# ...whereas plain K-Fold leaks massively
leaks_plain = [len(set(data['manager_id'].iloc[tr]) & set(data['manager_id'].iloc[te]))
               for tr, te in kfold_split(data, k=5)]
print("same for plain K-Fold                 :", leaks_plain)

fold   train   test  test %  managers
   0   38703   9676  20.0%       678
   1   38703   9676  20.0%       689
   2   38703   9676  20.0%       690
   3   38703   9676  20.0%       690
   4   38704   9675  20.0%       689

managers shared between train and test: [0, 0, 0, 0, 0]
same for plain K-Fold                 : [1827, 1838, 1820, 1830, 1838]


In [606]:
def stratified_kfold_split(df, k=5, stratify_field='bedrooms',
                           shuffle=True, random_state=RANDOM_STATE,
                           n_bins=10, merge_rare=True):
    """
    Stratified K-Fold: preserves the distribution of `stratify_field`
    in every fold (per-class test size is floor(n_c/k) or ceil(n_c/k)).

    Continuous fields are discretised into up to `n_bins` quantile bins.
    NaN is treated as its own class.
    Classes with fewer than k members are merged into the NEAREST class
    (label-wise) if merge_rare=True, otherwise an error is raised.

    Returns list of (train_idx, test_idx) with POSITIONAL indices -> use .iloc
    """
    n = len(df)
    if not isinstance(k, (int, np.integer)) or k < 2:
        raise ValueError("k must be an integer >= 2")
    if stratify_field not in df.columns:
        raise ValueError(f"column '{stratify_field}' not found")

    col = df[stratify_field]

    # ---- 1. build integer labels -------------------------------------------
    if pd.api.types.is_numeric_dtype(col) and col.nunique(dropna=True) > n_bins * 2:
        # bin by VALUES (not ranks) so equal values share a bin
        binned = pd.qcut(col, q=n_bins, labels=False, duplicates='drop')
        if binned.nunique(dropna=True) < 2:          # degenerate distribution
            binned = pd.qcut(col.rank(method='first'), q=n_bins,
                             labels=False, duplicates='drop')
        labels = binned.to_numpy(dtype=float)
    else:
        labels = pd.factorize(col, sort=True, use_na_sentinel=False)[0].astype(float)

    # NaN -> its own class, so no row is left unassigned
    nan_mask = np.isnan(labels)
    if nan_mask.any():
        labels[nan_mask] = np.nanmax(labels) + 1 if (~nan_mask).any() else 0
    labels = labels.astype(int)

    # ---- 2. handle rare classes (< k members) ------------------------------
    classes, counts = np.unique(labels, return_counts=True)
    if counts.min() < k:
        rare = classes[counts < k]
        if not merge_rare:
            raise ValueError(
                f"class(es) {rare.tolist()} have fewer than k={k} members "
                f"(min count = {counts.min()})")
        # repeatedly glue the smallest rare class into its nearest neighbour
        while True:
            classes, counts = np.unique(labels, return_counts=True)
            if counts.min() >= k or len(classes) == 1:
                break
            c = classes[np.argmin(counts)]
            others = classes[classes != c]
            target = others[np.argmin(np.abs(others - c))]   # nearest label
            labels[labels == c] = target
        classes, counts = np.unique(labels, return_counts=True)
        if counts.min() < k:
            raise ValueError(
                f"cannot stratify: only {n} rows / {counts.min()} in smallest "
                f"class for k={k}")

    # ---- 3. round-robin inside each class ----------------------------------
    rng = np.random.RandomState(random_state)
    test_folds = np.full(n, -1, dtype=int)

    # process big classes first and rotate the starting fold by the running
    # remainder -> leftovers are spread evenly instead of always hitting fold 0
    order = classes[np.argsort(-counts, kind='stable')]
    start = 0
    for c in order:
        cls_idx = np.flatnonzero(labels == c)
        if shuffle:
            rng.shuffle(cls_idx)
        test_folds[cls_idx] = (np.arange(len(cls_idx)) + start) % k
        start = (start + len(cls_idx)) % k

    if (test_folds < 0).any():
        raise RuntimeError("internal error: some rows were not assigned a fold")

    indices = np.arange(n)
    folds = [(indices[test_folds != f], indices[test_folds == f])
             for f in range(k)]

    assert sum(len(te) for _, te in folds) == n
    assert all(len(te) > 0 for _, te in folds)
    return folds

In [607]:
sfolds = stratified_kfold_split(data, k=5, stratify_field='bedrooms')

overall = data['bedrooms'].value_counts(normalize=True).sort_index()
rows = {'overall': overall}
for i, (_, te) in enumerate(sfolds):
    rows[f'fold {i}'] = data['bedrooms'].iloc[te].value_counts(
        normalize=True).sort_index()

print((pd.DataFrame(rows) * 100).round(2).fillna(0).to_string())

          overall  fold 0  fold 1  fold 2  fold 3  fold 4
bedrooms                                                 
0           19.02   19.02   19.03   19.02   19.02   19.02
1           32.12   32.12   32.12   32.12   32.12   32.12
2           30.07   30.07   30.06   30.06   30.06   30.07
3           14.60   14.59   14.59   14.60   14.59   14.59
4            3.68    3.68    3.67    3.67    3.68    3.68
5            0.43    0.42    0.43    0.42    0.42    0.42
6            0.09    0.07    0.09    0.10    0.10    0.08
7            0.00    0.00    0.00    0.00    0.00    0.01
8            0.00    0.02    0.00    0.00    0.00    0.00


In [608]:
def time_series_split(df, k=5, date_field=DATE, expanding=True, gap=0):
    """
    Time Series Split: train always precedes test in time.

    k        : number of folds (splits)
    gap      : number of rows skipped between train end and test start
    expanding: True  -> growing train window
               False -> rolling window of constant size

    Returns list of (train_idx, test_idx), len == k, ordered chronologically.
    """
    n = len(df)
    if not isinstance(k, (int, np.integer)) or k < 2:
        raise ValueError("k must be an integer >= 2")
    if date_field not in df.columns:
        raise ValueError(f"column '{date_field}' not found")
    if n < k + 1:
        raise ValueError("not enough rows for the requested number of folds")
    dates = pd.to_datetime(df[date_field]).to_numpy()
    order = np.argsort(dates, kind='stable')
    sorted_dates = dates[order]

    n_test = n // (k + 1)
    if n_test < 1:
        raise ValueError("k is too large for this dataset")

    folds = []
    for i in range(k):
        test_start = n - (k - i) * n_test
        test_stop = test_start + n_test if i < k - 1 else n

        while test_start > 0 and sorted_dates[test_start - 1] == sorted_dates[test_start]:
            test_start += 1
        train_stop = max(0, test_start - gap)
        if train_stop == 0:
            continue

        train_start = 0 if expanding else max(0, train_stop - n_test * (i + 1) // (i + 1) * (n_test))
        if not expanding:
            train_start = max(0, train_stop - n_test * 2)

        folds.append((order[train_start:train_stop], order[test_start:test_stop]))

    if not folds:
        raise ValueError("could not build any fold with the given parameters")
    return folds

In [609]:
tfolds = time_series_split(data, k=5, date_field=DATE)

print(f"{'fold':>4} {'train':>7} {'test':>6} "
      f"{'train period':>25} | {'test period':>25}   {'test mean':>9}")
for i, (tr, te) in enumerate(tfolds):
    dtr, dte = data[DATE].iloc[tr], data[DATE].iloc[te]
    print(f"{i:>4} {len(tr):>7} {len(te):>6} "
          f"{str(dtr.min().date()):>11} .. {str(dtr.max().date()):<11} | "
          f"{str(dte.min().date()):>11} .. {str(dte.max().date()):<11} "
          f"{y.iloc[te].mean():>9.2f}")

print("\nno future leakage (max train date < min test date):",
      all(data[DATE].iloc[tr].max() < data[DATE].iloc[te].min()
          for tr, te in tfolds))
print("train size grows monotonically:",
      all(len(tfolds[i][0]) < len(tfolds[i + 1][0]) for i in range(len(tfolds) - 1)))

fold   train   test              train period |               test period   test mean
   0    8064   8063  2016-04-01 .. 2016-04-16  |  2016-04-16 .. 2016-05-02    3544.90
   1   16127   8063  2016-04-01 .. 2016-05-02  |  2016-05-02 .. 2016-05-16    3522.52
   2   24190   8063  2016-04-01 .. 2016-05-16  |  2016-05-16 .. 2016-06-01    3604.54
   3   32253   8063  2016-04-01 .. 2016-06-01  |  2016-06-01 .. 2016-06-15    3536.23
   4   40316   8063  2016-04-01 .. 2016-06-15  |  2016-06-15 .. 2016-06-29    3553.16

no future leakage (max train date < min test date): True
train size grows monotonically: True


In [610]:
def cross_validate_custom(model_factory, X, y, folds, name=""):
    Xv, yv = np.asarray(X), np.asarray(y)
    rec = []
    for i, (tr, te) in enumerate(folds):
        m = model_factory().fit(Xv[tr], yv[tr])
        p_tr, p_te = m.predict(Xv[tr]), m.predict(Xv[te])
        rec.append({
            'fold': i,
            'MAE_train': mean_absolute_error(yv[tr], p_tr),
            'MAE_test':  mean_absolute_error(yv[te], p_te),
            'RMSE_test': mean_squared_error(yv[te], p_te) ** 0.5,
            'R2_test':   r2_score(yv[te], p_te),
        })
    df = pd.DataFrame(rec)
    return {
        'scheme': name,
        'MAE_train': df['MAE_train'].mean(),
        'MAE_test':  df['MAE_test'].mean(),
        'MAE_std':   df['MAE_test'].std(),
        'RMSE_test': df['RMSE_test'].mean(),
        'R2_test':   df['R2_test'].mean(),
    }


schemes = {
    'K-Fold':                 kfold_split(X, k=5),
    'GroupKFold(manager)':    group_kfold_split(data, k=5, group_field='manager_id'),
    'GroupKFold(building)':   group_kfold_split(data, k=5, group_field='building_id'),
    'StratifiedKFold(bed)':   stratified_kfold_split(data, k=5, stratify_field='bedrooms'),
    'TimeSeriesSplit':        time_series_split(data, k=5, date_field=DATE),
}

res = [cross_validate_custom(LinearRegression, X, y, f, n)
       for n, f in schemes.items()]
print(pd.DataFrame(res).round(3).to_string(index=False))

              scheme  MAE_train  MAE_test  MAE_std  RMSE_test  R2_test
              K-Fold   1030.797  1031.461    4.434   1455.878    0.169
 GroupKFold(manager)   1030.500  1035.446   35.148   1461.928    0.156
GroupKFold(building)   1030.554  1035.428   24.282   1458.630    0.160
StratifiedKFold(bed)   1030.832  1031.377    8.551   1455.809    0.170
     TimeSeriesSplit   1019.451  1029.626   15.367   1463.442    0.167


In [611]:
# ---------- 3.1 Подготовка таргета для стратификации ----------
# price - непрерывный, поэтому биним его на 10 квантильных бинов
price_bins = pd.qcut(y.rank(method='first'), q=10, labels=False, duplicates='drop')
bed_bins = data['bedrooms']

sk_schemes = {
    'K-Fold': list(KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE).split(X)),
    'GroupKFold(manager)': list(GroupKFold(n_splits=5).split(X, y, groups=data['manager_id'])),
    'GroupKFold(building)': list(GroupKFold(n_splits=5).split(X, y, groups=data['building_id'])),
    'StratifiedKFold(bed)': list(StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE).split(X, price_bins)),
    'TimeSeriesSplit': list(TimeSeriesSplit(n_splits=5).split(data.sort_values(DATE))),
}

sk_res = [cross_validate_custom(LinearRegression, X, y, f, n) for n, f in sk_schemes.items()]

print("=== MY IMPLEMENTATION ===")
print(pd.DataFrame(res).round(3).to_string(index=False))
print("\n=== SKLEARN ===")
print(pd.DataFrame(sk_res).round(3).to_string(index=False))

=== MY IMPLEMENTATION ===
              scheme  MAE_train  MAE_test  MAE_std  RMSE_test  R2_test
              K-Fold   1030.797  1031.461    4.434   1455.878    0.169
 GroupKFold(manager)   1030.500  1035.446   35.148   1461.928    0.156
GroupKFold(building)   1030.554  1035.428   24.282   1458.630    0.160
StratifiedKFold(bed)   1030.832  1031.377    8.551   1455.809    0.170
     TimeSeriesSplit   1019.451  1029.626   15.367   1463.442    0.167

=== SKLEARN ===
              scheme  MAE_train  MAE_test  MAE_std  RMSE_test  R2_test
              K-Fold   1030.797  1031.461    4.434   1455.878    0.169
 GroupKFold(manager)   1030.599  1034.585   45.334   1459.446    0.159
GroupKFold(building)   1030.348  1038.662   36.555   1461.127    0.152
StratifiedKFold(bed)   1030.807  1031.510    4.503   1455.796    0.170
     TimeSeriesSplit   1033.390  1035.284   15.100   1457.260    0.170


In [612]:
def train_feature_profile(folds, X, feature_cols):
    Xv = np.asarray(X)
    means = np.vstack([Xv[tr].mean(axis=0) for tr, _ in folds]).mean(axis=0)
    return pd.Series(means, index=feature_cols)

def compare_distributions(my_folds, sk_folds, X, feature_cols, name):
    a = train_feature_profile(my_folds, X, feature_cols)
    b = train_feature_profile(sk_folds, X, feature_cols)
    out = pd.DataFrame({'mine': a, 'sklearn': b})
    out['abs_diff'] = (out['mine'] - out['sklearn']).abs()
    out['rel_diff_%'] = (out['abs_diff'] / out['sklearn'].replace(0, np.nan) * 100)
    print(f"\n--- {name} (max abs diff = {out['abs_diff'].max():.5f}, "
        f"max rel diff = {out['rel_diff_%'].max():.3f}%) ---")
    print(out.round(4).to_string())
    return out

for name in schemes:
    compare_distributions(schemes[name], sk_schemes[name], X, TOP20, name)


--- K-Fold (max abs diff = 0.00000, max rel diff = 0.000%) ---
                     mine  sklearn  abs_diff  rel_diff_%
Elevator           0.5321   0.5321       0.0         0.0
HardwoodFloors     0.4793   0.4793       0.0         0.0
CatsAllowed        0.4785   0.4785       0.0         0.0
DogsAllowed        0.4478   0.4478       0.0         0.0
Doorman            0.4251   0.4251       0.0         0.0
Dishwasher         0.4232   0.4232       0.0         0.0
LaundryinBuilding  0.3795   0.3795       0.0         0.0
NoFee              0.3681   0.3681       0.0         0.0
FitnessCenter      0.2687   0.2687       0.0         0.0
LaundryinUnit      0.1870   0.1870       0.0         0.0
Pre-War            0.1856   0.1856       0.0         0.0
RoofDeck           0.1341   0.1341       0.0         0.0
OutdoorSpace       0.1062   0.1062       0.0         0.0
DiningRoom         0.1016   0.1016       0.0         0.0
HighSpeedInternet  0.0873   0.0873       0.0         0.0
balcony            0.061

In [613]:
cmp = pd.DataFrame(res).set_index('scheme')[['MAE_test', 'MAE_std', 'R2_test']]
cmp['gap_train_test'] = (pd.DataFrame(res).set_index('scheme')['MAE_test']
                         - pd.DataFrame(res).set_index('scheme')['MAE_train'])
print(cmp.round(3).to_string())

                      MAE_test  MAE_std  R2_test  gap_train_test
scheme                                                          
K-Fold                1031.461    4.434    0.169           0.664
GroupKFold(manager)   1035.446   35.148    0.156           4.946
GroupKFold(building)  1035.428   24.282    0.160           4.874
StratifiedKFold(bed)  1031.377    8.551    0.170           0.545
TimeSeriesSplit       1029.626   15.367    0.167          10.175


In [614]:
FEATURES = TOP20 + ['bedrooms', 'bathrooms']

work = data[[DATE, TARGET] + FEATURES].copy()

q_val, q_test = work[DATE].quantile([0.60, 0.80])

X_tr, X_val, X_te, y_tr, y_val, y_te = train_val_test_split_by_date(
    work, validation_date=q_val, test_date=q_test,
    date_column=DATE, target_column=TARGET, feature_columns=FEATURES
)

print(FEATURES)

['Elevator', 'HardwoodFloors', 'CatsAllowed', 'DogsAllowed', 'Doorman', 'Dishwasher', 'LaundryinBuilding', 'NoFee', 'FitnessCenter', 'LaundryinUnit', 'Pre-War', 'RoofDeck', 'OutdoorSpace', 'DiningRoom', 'HighSpeedInternet', 'balcony', 'SwimmingPool', 'NewConstruction', 'Terrace', 'Exclusive', 'bedrooms', 'bathrooms']


In [615]:
def make_lasso(alpha=0.1):
    return Pipeline([('scaler', MinMaxScaler()),
                     ('model', Lasso(alpha=alpha, random_state=RANDOM_STATE,
                                     max_iter=10000))])


def evaluate(model, feats, tag=""):
    t0 = time.time()
    model.fit(X_tr[feats], y_tr)
    fit_time = time.time() - t0
    out = {'model': tag, 'fit_time_s': fit_time}
    for part, (Xp, yp) in {'train': (X_tr, y_tr), 'val': (X_val, y_val), 'test': (X_te, y_te)}.items():
        p = model.predict(Xp[feats])
        out[f'MAE_{part}'] = mean_absolute_error(yp, p)
        out[f'RMSE_{part}'] = mean_squared_error(yp, p) ** 0.5
        out[f'R2_{part}'] = r2_score(yp, p)
    return out


baseline = evaluate(make_lasso(), FEATURES, "Lasso / all features")
print(pd.Series(baseline).to_string())

model         Lasso / all features
fit_time_s                0.064554
MAE_train               705.426117
RMSE_train              1032.32145
R2_train                  0.576689
MAE_val                 721.421924
RMSE_val               1049.507759
R2_val                    0.587477
MAE_test                724.287067
RMSE_test                1043.8687
R2_test                   0.570422


In [616]:
lasso_full = make_lasso().fit(X_tr[FEATURES], y_tr)
coefs = pd.Series(lasso_full.named_steps['model'].coef_, index=FEATURES)

lasso_rank = coefs.abs().sort_values(ascending=False)

print(lasso_rank)
top10_lasso = lasso_rank.head(10).index.tolist()
print("\nTOP-10 by |coef|:", top10_lasso)

bathrooms            15479.378507
bedrooms              3214.578671
Doorman                623.382066
LaundryinUnit          477.108957
LaundryinBuilding      246.862153
FitnessCenter          238.324169
HighSpeedInternet      225.550173
Elevator               201.577565
Terrace                185.342589
NoFee                  162.741532
DogsAllowed            159.326472
NewConstruction        122.192446
Dishwasher             118.649312
balcony                116.097061
DiningRoom             115.993180
CatsAllowed             98.441632
HardwoodFloors          91.928874
RoofDeck                89.695786
OutdoorSpace            68.536425
Exclusive               54.036341
SwimmingPool            14.425148
Pre-War                 10.984901
dtype: float64

TOP-10 by |coef|: ['bathrooms', 'bedrooms', 'Doorman', 'LaundryinUnit', 'LaundryinBuilding', 'FitnessCenter', 'HighSpeedInternet', 'Elevator', 'Terrace', 'NoFee']


In [617]:
rows = [
    evaluate(make_lasso(), FEATURES, tag='Lasso MinMaxScaler'),
    evaluate(make_lasso(), top10_lasso, tag="Lasso top10 MinMaxScaler"),
]
results = pd.DataFrame(rows).set_index('model')

mae_table = results[['MAE_train', 'MAE_val', 'MAE_test']]
print(mae_table.round(4))

RMSE_table = results[['RMSE_train', 'RMSE_val', 'RMSE_test']]
print(RMSE_table.round(4))

R2_table = results[['R2_train', 'R2_val', 'R2_test']]
print(R2_table.round(4))

                          MAE_train   MAE_val  MAE_test
model                                                  
Lasso MinMaxScaler         705.4261  721.4219  724.2871
Lasso top10 MinMaxScaler   707.8075  724.9707  728.5758
                          RMSE_train   RMSE_val  RMSE_test
model                                                     
Lasso MinMaxScaler         1032.3215  1049.5078  1043.8687
Lasso top10 MinMaxScaler   1035.8176  1053.8511  1047.5403
                          R2_train  R2_val  R2_test
model                                              
Lasso MinMaxScaler          0.5767  0.5875   0.5704
Lasso top10 MinMaxScaler    0.5738  0.5841   0.5674


In [618]:
def simple_feature_selection(X_df, y_ser, top_k=10,
                             nan_threshold=0.3, corr_threshold=0.9):
    """
    1) убираем признаки с долей NaN > nan_threshold;
    2) убираем константы;
    3) убираем один из пары признаков с |corr| > corr_threshold
       (оставляем тот, у кого корреляция с таргетом выше);
    4) ранжируем оставшиеся по |corr(feature, target)| и берём top_k.
    Возвращает (список признаков, отчёт).
    """
    X_df = X_df.copy()
    nan_ratio = X_df.isna().mean()
    kept = nan_ratio[nan_ratio <= nan_threshold].index.tolist()
    dropped_nan = [c for c in X_df.columns if c not in kept]

    nunq = X_df[kept].nunique()
    const = nunq[nunq <= 1].index.tolist()
    kept = [c for c in kept if c not in const]

    target_corr = X_df[kept].corrwith(y_ser).abs().fillna(0)

    corr_mat = X_df[kept].corr().abs()
    dropped_corr = set()
    order = target_corr.sort_values(ascending=False).index.tolist()
    for i, a in enumerate(order):
        if a in dropped_corr:
            continue
        for b in order[i + 1:]:
            if b in dropped_corr:
                continue
            if corr_mat.loc[a, b] > corr_threshold:
                dropped_corr.add(b) 
    kept = [c for c in order if c not in dropped_corr]

    selected = target_corr[kept].sort_values(ascending=False).head(top_k).index.tolist()
    report = {
        'dropped_by_nan': dropped_nan,
        'dropped_const': const,
        'dropped_by_corr': sorted(dropped_corr),
        'target_corr': target_corr.sort_values(ascending=False),
    } 
    return selected, report

top10_simple, rep = simple_feature_selection(X_tr, y_tr, top_k=10)
print("dropped by nan  :", rep['dropped_by_nan'])
print("dropped const   :", rep['dropped_const'])
print("dropped by corr :", rep['dropped_by_corr'])
print("\ncorr with target:\n", rep['target_corr'].round(3).to_string())
print("\nTOP-10 simple:", top10_simple)

r_simple = evaluate(make_lasso(), top10_simple, "NaN+Corr TOP-10")
print(pd.Series(r_simple).to_string())

dropped by nan  : []
dropped const   : []
dropped by corr : ['CatsAllowed']

corr with target:
 bathrooms            0.669
bedrooms             0.550
Doorman              0.274
LaundryinUnit        0.274
FitnessCenter        0.231
Dishwasher           0.220
DiningRoom           0.220
Elevator             0.207
OutdoorSpace         0.150
NoFee                0.133
Terrace              0.133
RoofDeck             0.124
SwimmingPool         0.124
LaundryinBuilding    0.123
balcony              0.117
HardwoodFloors       0.101
HighSpeedInternet    0.096
DogsAllowed          0.067
NewConstruction      0.063
CatsAllowed          0.057
Pre-War              0.028
Exclusive            0.007

TOP-10 simple: ['bathrooms', 'bedrooms', 'Doorman', 'LaundryinUnit', 'FitnessCenter', 'Dishwasher', 'DiningRoom', 'Elevator', 'OutdoorSpace', 'NoFee']
model         NaN+Corr TOP-10
fit_time_s           0.010561
MAE_train          715.802375
RMSE_train        1044.119126
R2_train             0.566958
MAE_val 

In [619]:
def permutation_importance_custom(model, X_df, y_ser, scoring=mean_absolute_percentage_error,
                                  n_repeats=5, random_state=RANDOM_STATE,
                                  greater_is_better=False):
    """
    Важность = (ошибка после перемешивания признака) - (базовая ошибка).
    Чем больше значение, тем важнее признак.
    """
    rng = np.random.RandomState(random_state)
    base = scoring(y_ser, model.predict(X_df))
    sign = -1.0 if greater_is_better else 1.0

    imp_mean, imp_std = {}, {}
    for col in X_df.columns:
        original = X_df[col].to_numpy().copy()
        scores = []
        for _ in range(n_repeats):
            shuffled = original.copy()
            rng.shuffle(shuffled)
            X_df[col] = shuffled
            scores.append(scoring(y_ser, model.predict(X_df)))
        X_df[col] = original
        deltas = sign * (np.array(scores) - base)
        imp_mean[col] = deltas.mean()
        imp_std[col] = deltas.std()

    return (pd.DataFrame({'importance': pd.Series(imp_mean),
                          'std': pd.Series(imp_std)})).sort_values('importance', ascending=False)

t0 = time.time()
perm = permutation_importance_custom(lasso_full, X_val.copy(), y_val, n_repeats=5)
perm_time = time.time()- t0
print(f"permutation importance time: {perm_time:.2f}s")
print(perm.round(3).to_string())

permutation importance time: 0.32s
                   importance    std
bathrooms               0.080  0.002
bedrooms                0.070  0.001
Doorman                 0.033  0.000
LaundryinUnit           0.010  0.001
Elevator                0.005  0.000
FitnessCenter           0.005  0.000
LaundryinBuilding       0.003  0.000
DogsAllowed             0.002  0.000
Dishwasher              0.002  0.000
NoFee                   0.001  0.000
HighSpeedInternet       0.001  0.000
Terrace                 0.001  0.000
CatsAllowed             0.001  0.000
HardwoodFloors          0.000  0.000
DiningRoom              0.000  0.000
NewConstruction         0.000  0.000
Exclusive               0.000  0.000
SwimmingPool            0.000  0.000
RoofDeck                0.000  0.000
Pre-War                 0.000  0.000
OutdoorSpace           -0.000  0.000
balcony                -0.000  0.000


In [620]:
top10_perm = perm.head(10).index.tolist()

rows = [
    evaluate(make_lasso(), FEATURES, tag='Lasso MinMaxScaler'),
    evaluate(make_lasso(), top10_lasso, tag="Lasso top10 MinMaxScaler"),
    evaluate(make_lasso(), top10_perm, "Lasso Permutation MinMaxScaler"),
]
results = pd.DataFrame(rows).set_index('model')

mae_table = results[['MAE_train', 'MAE_val', 'MAE_test']]
print(mae_table.round(4))

RMSE_table = results[['RMSE_train', 'RMSE_val', 'RMSE_test']]
print(RMSE_table.round(4))

R2_table = results[['R2_train', 'R2_val', 'R2_test']]
print(R2_table.round(4))

                                MAE_train   MAE_val  MAE_test
model                                                        
Lasso MinMaxScaler               705.4261  721.4219  724.2871
Lasso top10 MinMaxScaler         707.8075  724.9707  728.5758
Lasso Permutation MinMaxScaler   709.7667  726.7722  728.2872
                                RMSE_train   RMSE_val  RMSE_test
model                                                           
Lasso MinMaxScaler               1032.3215  1049.5078  1043.8687
Lasso top10 MinMaxScaler         1035.8176  1053.8511  1047.5403
Lasso Permutation MinMaxScaler   1037.8420  1055.7038  1047.8429
                                R2_train  R2_val  R2_test
model                                                    
Lasso MinMaxScaler                0.5767  0.5875   0.5704
Lasso top10 MinMaxScaler          0.5738  0.5841   0.5674
Lasso Permutation MinMaxScaler    0.5721  0.5826   0.5671


In [621]:
t0 = time.time()
X_tr_scaled = lasso_full.named_steps['scaler'].transform(X_tr[FEATURES])
explainer = shap.LinearExplainer(lasso_full.named_steps['model'], masker=shap.maskers.Independent(X_tr_scaled, max_samples=len(X_tr_scaled)))
shap_values = explainer.shap_values(
    lasso_full.named_steps['scaler'].transform(X_val[FEATURES]))
shap_time = time.time() - t0

shap_imp = pd.Series(np.abs(shap_values).mean(axis=0),
                     index=FEATURES).sort_values(ascending=False)
print(f"shap time: {shap_time:.2f}s")
print(shap_imp.round(3).to_string())

top10_shap = shap_imp.head(10).index.tolist()

shap time: 0.01s
bathrooms            514.941
bedrooms             421.170
Doorman              304.555
LaundryinUnit        145.006
LaundryinBuilding    116.154
Elevator             100.347
FitnessCenter         93.061
DogsAllowed           78.902
NoFee                 75.612
Dishwasher            57.960
CatsAllowed           49.169
HardwoodFloors        45.896
HighSpeedInternet     35.521
DiningRoom            21.149
RoofDeck              20.749
Terrace               16.386
balcony               13.488
OutdoorSpace          12.978
NewConstruction       12.025
Exclusive              4.638
Pre-War                3.371
SwimmingPool           1.458


In [622]:
rows = [
    evaluate(make_lasso(), FEATURES, tag='Lasso MinMaxScaler'),
    evaluate(make_lasso(), top10_lasso, tag="Lasso top10 MinMaxScaler"),
    evaluate(make_lasso(), top10_perm, "Lasso Permutation MinMaxScaler"),
    evaluate(make_lasso(), top10_shap, "Lasso shap MinMaxScaler")
]
results = pd.DataFrame(rows).set_index('model')

mae_table = results[['MAE_train', 'MAE_val', 'MAE_test']]
print(mae_table.round(4))

RMSE_table = results[['RMSE_train', 'RMSE_val', 'RMSE_test']]
print(RMSE_table.round(4))

R2_table = results[['R2_train', 'R2_val', 'R2_test']]
print(R2_table.round(4))

                                MAE_train   MAE_val  MAE_test
model                                                        
Lasso MinMaxScaler               705.4261  721.4219  724.2871
Lasso top10 MinMaxScaler         707.8075  724.9707  728.5758
Lasso Permutation MinMaxScaler   709.7667  726.7722  728.2872
Lasso shap MinMaxScaler          709.7668  726.7722  728.2873
                                RMSE_train   RMSE_val  RMSE_test
model                                                           
Lasso MinMaxScaler               1032.3215  1049.5078  1043.8687
Lasso top10 MinMaxScaler         1035.8176  1053.8511  1047.5403
Lasso Permutation MinMaxScaler   1037.8420  1055.7038  1047.8429
Lasso shap MinMaxScaler          1037.8420  1055.7039  1047.8429
                                R2_train  R2_val  R2_test
model                                                    
Lasso MinMaxScaler                0.5767  0.5875   0.5704
Lasso top10 MinMaxScaler          0.5738  0.5841   0.5674
Lasso 

In [623]:
# --- стабильность: пересечение TOP-10 на бутстрап-подвыборках train ---
def stability(selector, n_runs=5, frac=0.7):
    sets = []
    for s in range(n_runs):
        idx = X_tr.sample(frac=frac, random_state=s).index
        sets.append(set(selector(X_tr.loc[idx], y_tr.loc[idx])))
    jac = [len(a & b) / len(a | b)
           for i, a in enumerate(sets) for b in sets[i + 1:]]
    return np.mean(jac)


def sel_lasso(Xs, ys):
    m = make_lasso().fit(Xs[FEATURES], ys)
    c = pd.Series(m.named_steps['model'].coef_, index=FEATURES).abs()
    return c.sort_values(ascending=False).head(10).index.tolist()


def sel_simple(Xs, ys):
    return simple_feature_selection(Xs, ys, top_k=10)[0]


def sel_perm(Xs, ys):
    m = make_lasso().fit(Xs[FEATURES], ys)
    p = permutation_importance_custom(m, X_val.copy(), y_val, n_repeats=3)
    return p.head(10).index.tolist()


def sel_shap(Xs, ys):
    m = make_lasso().fit(Xs[FEATURES], ys)
    sc = m.named_steps['scaler']
    ex = shap.LinearExplainer(m.named_steps['model'], masker=shap.maskers.Independent(sc.transform(Xs[FEATURES]), max_samples=len(Xs)))
    sv = ex.shap_values(sc.transform(X_val[FEATURES]))
    s = pd.Series(np.abs(sv).mean(axis=0), index=FEATURES)
    return s.sort_values(ascending=False).head(10).index.tolist()


stab = pd.Series({
    'Lasso coef TOP-10': stability(sel_lasso),
    'NaN+Corr TOP-10': stability(sel_simple),
    'Permutation TOP-10': stability(sel_perm),
    'SHAP TOP-10': stability(sel_shap),
}, name='stability_jaccard')
print(stab.round(3).to_string())

Lasso coef TOP-10     0.855
NaN+Corr TOP-10       0.891
Permutation TOP-10    0.891
SHAP TOP-10           1.000


In [624]:
FS = top10_shap

def make_enet(alpha, l1_ratio):
    return Pipeline([('scaler', StandardScaler()),
                     ('model', ElasticNet(alpha=alpha, l1_ratio=l1_ratio, random_state=RANDOM_STATE, max_iter=10000))])

def score_params(alpha, l1_ratio):
    """Обучаем на train, метрика на validation (MAE, чем меньше - тем лучше)."""
    m = make_enet(alpha, l1_ratio).fit(X_tr[FS], y_tr)
    return mean_absolute_error(y_val, m.predict(X_val[FS]))

In [625]:
def grid_search_custom(alphas, l1_ratios, score_fn=score_params):
    rows = []
    for a, l1 in product(alphas, l1_ratios):
        rows.append({'alpha': a, 'l1_ratio': l1, 'MAE_val': score_fn(a, l1)})
    res = pd.DataFrame(rows).sort_values('MAE_val').reset_index(drop=True)
    best = res.iloc[0]
    return {'alpha': best['alpha'], 'l1_ratio': best['l1_ratio']}, best['MAE_val'], res

alphas = np.logspace(-3, 2, 12)
l1_ratios = np.linspace(0.05, 1.0, 8)

t0 = time.time()
best_grid, mae_grid, grid_table = grid_search_custom(alphas, l1_ratios)
t_grid = time.time() - t0

print(f"GridSearch: {len(grid_table)} combos in {t_grid:.2f}s")
print("best:", best_grid, "MAE_val =", round(mae_grid, 3))
print(grid_table.head(10).round(4).to_string(index=False))

GridSearch: 96 combos in 0.84s
best: {'alpha': np.float64(0.1873817422860383), 'l1_ratio': np.float64(0.8642857142857143)} MAE_val = 726.55
 alpha  l1_ratio  MAE_val
0.1874    0.8643 726.5500
0.0658    0.5929 726.5516
0.0231    0.0500 726.5588
0.0231    0.1857 726.5741
0.0658    0.4571 726.5745
0.0658    0.7286 726.5782
0.0231    0.3214 726.5929
0.0231    0.4571 726.6142
4.3288    1.0000 726.6336
0.0658    0.3214 726.6343


In [626]:
def random_search_custom(n_iter=40, random_state=RANDOM_STATE, score_fn=score_params):
    rng = np.random.RandomState(random_state)
    rows = []
    for _ in range(n_iter):
        a = 10 ** rng.uniform(-3, 2)
        l1 = rng.uniform(0.01, 1.0)
        rows.append({'alpha': a, 'l1_ratio': l1, 'MAE_val': score_fn(a, l1)})
    res = pd.DataFrame(rows).sort_values('MAE_val').reset_index(drop=True)
    best = res.iloc[0]
    return {'alpha': best['alpha'], 'l1_ratio': best['l1_ratio']}, best['MAE_val'], res

t0 = time.time()
best_rand, mae_rand, rand_table = random_search_custom(n_iter=40)
t_rand = time.time() - t0

print(f"RandomSearch: 40 trials in {t_rand:.2f}s")
print("best:", {k: round(v, 4) for k, v in best_rand.items()},
      "MAE_val =", round(mae_rand, 3))
print(rand_table.head(10).round(4).to_string(index=False))

RandomSearch: 40 trials in 0.37s
best: {'alpha': np.float64(0.0433), 'l1_ratio': np.float64(0.4655)} MAE_val = 726.555
 alpha  l1_ratio  MAE_val
0.0433    0.4655 726.5553
0.0312    0.2911 726.5582
0.1106    0.7162 726.5587
0.0347    0.5878 726.6015
0.3018    0.8651 726.6047
0.0325    0.6673 726.6285
0.0107    0.0603 726.6360
0.0339    0.8219 726.6784
0.0143    0.5946 726.6821
0.0069    0.1875 726.6849


In [627]:
candidates = {'grid': (best_grid, mae_grid), 'random': (best_rand, mae_rand)}
winner = min(candidates, key=lambda k: candidates[k][1])
best_params = candidates[winner][0]
print('winner:', winner, best_params)

final_model = make_enet(**best_params).fit(X_tr[FS], y_tr)

final_metrics = {}
for part, (Xp, yp) in {'train': (X_tr, y_tr), 'val': (X_val, y_val),
                       'test': (X_te, y_te)}.items():
    p = final_model.predict(Xp[FS])
    final_metrics[part] = {'MAE': mean_absolute_error(yp, p),
                           'RMSE': mean_squared_error(yp, p) ** 0.5,
                           'R2': r2_score(yp, p)}
print(pd.DataFrame(final_metrics).T.round(3).to_string())

X_full = pd.concat([X_tr, X_val])
y_full = pd.concat([y_tr, y_val])
final_model_full = make_enet(**best_params).fit(X_full[FS], y_full)
p = final_model_full.predict(X_te[FS])
print("refit on train+val -> test MAE:", round(mean_absolute_error(y_te, p), 3),
      "| R2:", round(r2_score(y_te, p), 3))

winner: grid {'alpha': np.float64(0.1873817422860383), 'l1_ratio': np.float64(0.8642857142857143)}
           MAE      RMSE     R2
train  709.415  1038.076  0.572
val    726.550  1056.777  0.582
test   728.314  1048.660  0.566
refit on train+val -> test MAE: 729.42 | R2: 0.567


In [628]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    alpha = trial.suggest_float('alpha', 1e-3, 1e2, log=True)
    l1_ratio = trial.suggest_float('l1_ratio', 0.01, 1.0)
    return score_params(alpha, l1_ratio)

t0 = time.time()
study = optuna.create_study(direction='minimize',
                            sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study.optimize(objective, n_trials=40)
t_optuna = time.time() - t0

print(f"Optuna: 40 trials in {t_optuna:.2f}s")
print("best params:", study.best_params, "MAE_val =", round(study.best_value, 3))

opt_model = make_enet(**study.best_params).fit(X_tr[FS], y_tr)

Optuna: 40 trials in 0.50s
best params: {'alpha': 0.05091365411920243, 'l1_ratio': 0.4431760851268408} MAE_val = 726.553


In [629]:
def test_metrics(params, tag, n_evals, elapsed, mae_val):
    m = make_enet(**params).fit(X_tr[FS], y_tr)
    p = m.predict(X_te[FS])
    return {'search': tag, 'n_evals': n_evals, 'time_s': elapsed,
            'alpha': params['alpha'], 'l1_ratio': params['l1_ratio'],
            'MAE_val': mae_val,
            'MAE_test': mean_absolute_error(y_te, p),
            'RMSE_test': mean_squared_error(y_te, p) ** 0.5,
            'R2_test': r2_score(y_te, p)}


comparison = pd.DataFrame([
    test_metrics(best_grid, 'GridSearch (mine)', len(grid_table), t_grid, mae_grid),
    test_metrics(best_rand, 'RandomSearch (mine)', len(rand_table), t_rand, mae_rand),
    test_metrics(study.best_params, 'Optuna TPE', 40, t_optuna, study.best_value),
])
print(comparison.round(4).to_string(index=False))

             search  n_evals  time_s  alpha  l1_ratio  MAE_val  MAE_test  RMSE_test  R2_test
  GridSearch (mine)       96  0.8422 0.1874    0.8643 726.5500  728.3138  1048.6605   0.5665
RandomSearch (mine)       40  0.3742 0.0433    0.4655 726.5553  728.2964  1048.5542   0.5666
         Optuna TPE       40  0.5037 0.0509    0.4432 726.5529  728.3482  1048.7749   0.5664


In [630]:
data_fs = data.reset_index(drop=True)
cv_folds = time_series_split(data_fs, k=5, date_field=DATE)

X_cv = data_fs[FS]
y_cv = data_fs[TARGET]


def objective_cv(trial):
    alpha = trial.suggest_float('alpha', 1e-3, 1e2, log=True)
    l1_ratio = trial.suggest_float('l1_ratio', 0.01, 1.0)

    Xv, yv = X_cv.to_numpy(), y_cv.to_numpy()
    scores = []
    for tr, te in cv_folds:
        m = make_enet(alpha, l1_ratio).fit(Xv[tr], yv[tr])
        scores.append(mean_absolute_error(yv[te], m.predict(Xv[te])))
    trial.set_user_attr('mae_std', float(np.std(scores)))
    return float(np.mean(scores))


t0 = time.time()
study_cv = optuna.create_study(direction='minimize',
                               sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_cv.optimize(objective_cv, n_trials=30)
t_cv = time.time() - t0

print(f"Optuna + TimeSeriesSplit: 30 trials in {t_cv:.2f}s")
print("best params:", study_cv.best_params)
print("CV MAE =", round(study_cv.best_value, 3),
      "+/-", round(study_cv.best_trial.user_attrs['mae_std'], 3))

# финальная модель: обучаем на train+val, честно оцениваем на test
final_cv_model = make_enet(**study_cv.best_params).fit(X_full[FS], y_full)
p = final_cv_model.predict(X_te[FS])
print("test MAE:", round(mean_absolute_error(y_te, p), 3),
      "| test RMSE:", round(mean_squared_error(y_te, p) ** 0.5, 3),
      "| test R2:", round(r2_score(y_te, p), 3))

print(optuna.importance.get_param_importances(study_cv))

Optuna + TimeSeriesSplit: 30 trials in 1.60s
best params: {'alpha': 0.11602135867140675, 'l1_ratio': 0.806988484076279}
CV MAE = 717.812 +/- 10.185
test MAE: 729.402 | test RMSE: 1047.804 | test R2: 0.567
{'alpha': np.float64(0.7617098564120238), 'l1_ratio': np.float64(0.23829014358797612)}


In [631]:
# сводная таблица по всем стратегиям подбора
comparison_all = pd.concat([
    comparison,
    pd.DataFrame([test_metrics(study_cv.best_params, 'Optuna + TimeSeriesSplit',
                               30, t_cv, study_cv.best_value)])
], ignore_index=True)
print(comparison_all.round(4).to_string(index=False))

                  search  n_evals  time_s  alpha  l1_ratio  MAE_val  MAE_test  RMSE_test  R2_test
       GridSearch (mine)       96  0.8422 0.1874    0.8643 726.5500  728.3138  1048.6605   0.5665
     RandomSearch (mine)       40  0.3742 0.0433    0.4655 726.5553  728.2964  1048.5542   0.5666
              Optuna TPE       40  0.5037 0.0509    0.4432 726.5529  728.3482  1048.7749   0.5664
Optuna + TimeSeriesSplit       30  1.6024 0.1160    0.8070 717.8121  728.2876  1048.5301   0.5666
